# 04 — Composição dos Datasets

Este notebook implementa a **Issue 1 — Compor datasets e criar base final de vagas**.

O objetivo é criar uma base final de vagas enriquecidas a partir de:

- LinkedIn Job Postings 2023-2024;
- Job Skill Set Dataset.

A base Resume-JD-Match não entra diretamente na composição de vagas, pois seu papel no projeto é treinar o modelo supervisionado `Fit / No Fit`.

## Decisões de composição

A estratégia adotada foi:

1. Usar o **LinkedIn Job Postings** como catálogo principal de vagas reais.
2. Usar o **Job Skill Set** como enriquecimento de skills.
3. Fazer a composição usando `job_id`, quando houver correspondência entre as bases.
4. Usar `left join`, preservando as vagas do LinkedIn mesmo quando não houver correspondência no Job Skill Set.
5. Unificar skills vindas do próprio LinkedIn e do Job Skill Set.
6. Criar a coluna `texto_vaga_completo`, juntando cargo, descrição e skills.
7. Salvar a base completa em `data/processed/vagas_enriquecidas.csv`.
8. Criar uma amostra pequena versionável em `data/sample/vagas_exemplo.csv` para os outros desenvolvedores testarem seus módulos.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import os
import numpy as np


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)

Raiz do projeto: /home/lucas/projetos-python/jobmatch-ai


In [3]:
from src.compose_datasets import (
    COLUNAS_BASE_FINAL,
    preparar_linkedin_jobs,
    preparar_job_skill_set,
    verificar_intersecao_job_id,
    compor_datasets_vagas,
    validar_base_final,
    salvar_base_final,
    criar_amostra_versionavel,
)

## 1. Carregar e preparar LinkedIn Job Postings

Esta etapa carrega o dataset do LinkedIn, seleciona as colunas principais e agrega as skills mapeadas pelos arquivos auxiliares `job_skills.csv` e `skills.csv`, quando disponíveis.

In [4]:
df_linkedin = preparar_linkedin_jobs(
    project_root=PROJECT_ROOT,
    usar_kagglehub=True
)

print("Formato LinkedIn preparado:", df_linkedin.shape)
display(df_linkedin.head())

Formato LinkedIn preparado: (123849, 16)


,job_id,title,title_normalized,description,company_name,location,min_salary,med_salary,max_salary,pay_period,formatted_work_type,formatted_experience_level,remote_allowed,job_posting_url,skills_linkedin,skills_linkedin_lista
0,921716,Marketing Coordinator,marketing coordinator,Job descriptionA leading real estate firm in N...,Corcoran Sawyer Smith,"Princeton, NJ",17.0,NaN,20.0,HOURLY,Full-time,NaN,NaN,https://www.linkedin.com/jobs/view/921716/?trk...,marketing; sales,"[marketing, sales]"
1,1829192,Mental Health Therapist/Counselor,mental health therapist counselor,"At Aspen Therapy and Wellness , we are committ...",NaN,"Fort Collins, CO",30.0,NaN,50.0,HOURLY,Full-time,NaN,NaN,https://www.linkedin.com/jobs/view/1829192/?tr...,health care provider,[health care provider]
2,10998357,Assitant Restaurant Manager,assitant restaurant manager,The National Exemplar is accepting application...,The National Exemplar,"Cincinnati, OH",45000.0,NaN,65000.0,YEARLY,Full-time,NaN,NaN,https://www.linkedin.com/jobs/view/10998357/?t...,management; manufacturing,"[management, manufacturing]"
3,23221523,Senior Elder Law / Trusts and Estates Associat...,senior elder law trusts and estates associate ...,Senior Associate Attorney - Elder Law / Trusts...,"Abrams Fensterman, LLP","New Hyde Park, NY",140000.0,NaN,175000.0,YEARLY,Full-time,NaN,NaN,https://www.linkedin.com/jobs/view/23221523/?t...,other,[other]
4,35982263,Service Technician,service technician,Looking for HVAC service tech with experience ...,NaN,"Burlington, IA",60000.0,NaN,80000.0,YEARLY,Full-time,NaN,NaN,https://www.linkedin.com/jobs/view/35982263/?t...,information technology,[information technology]


In [5]:
print("Colunas LinkedIn preparado:")
print(df_linkedin.columns.tolist())

print("Job IDs duplicados no LinkedIn preparado:")
print(df_linkedin["job_id"].duplicated().sum())

print("Vagas com skills do LinkedIn:")
print(df_linkedin["skills_linkedin"].fillna("").astype(str).str.strip().ne("").sum())

Colunas LinkedIn preparado:
['job_id', 'title', 'title_normalized', 'description', 'company_name', 'location', 'min_salary', 'med_salary', 'max_salary', 'pay_period', 'formatted_work_type', 'formatted_experience_level', 'remote_allowed', 'job_posting_url', 'skills_linkedin', 'skills_linkedin_lista']
Job IDs duplicados no LinkedIn preparado:
0
Vagas com skills do LinkedIn:
122096


## 2. Carregar e preparar Job Skill Set

Esta etapa carrega o Job Skill Set e transforma a coluna `job_skill_set` em uma lista normalizada de skills.

In [6]:
df_job_skill = preparar_job_skill_set(
    project_root=PROJECT_ROOT,
    usar_kagglehub=True
)

print("Formato Job Skill Set preparado:", df_job_skill.shape)
display(df_job_skill.head())

100%|██████████| 1.50M/1.50M [00:00<00:00, 1.81MB/s]

Extracting files...


Formato Job Skill Set preparado: (1167, 7)


,job_id,category,job_title,title_normalized,job_description,skills_job_skill_set,skills_job_skill_set_lista
0,3651735073,FINANCE,Director of Finance,director of finance,APPLY HERE\nHTTPS://WWW.INDEED.COM/JOB/DIRECTO...,accounting; accounts payable; analytical skill...,"[accounting, accounts payable, analytical skil..."
1,3715891138,HR,Human Resources Manager,human resources manager,HUMAN RESOURCES MANAGER POSITION SUMMARYTHE I...,adaptability; applicant tracking systems; bene...,"[adaptability, applicant tracking systems, ben..."
2,3782812525,SALES,Salesperson,salesperson,THE TERRA FORZA GOLF TEAM IS GROWING! TERRITOR...,club fitting; communication; computer skills; ...,"[club fitting, communication, computer skills,..."
3,3861704803,BUSINESS-DEVELOPMENT,Business Development Manager,business development manager,"HELLO FOLKS,HOPE YOU ARE WELL AND DOING GREAT,...",account management; aggressive; articulate; bu...,"[account management, aggressive, articulate, b..."
4,3862673730,HR,AOC Human Resources Project Manager,aoc human resources project manager,ADMINISTRATIVE OFFICE OF THE COURTSBUSINESS UN...,adaptability; analytical skills; attention to ...,"[adaptability, analytical skills, attention to..."


In [7]:
print("Colunas Job Skill Set preparado:")
print(df_job_skill.columns.tolist())

print("Job IDs duplicados no Job Skill Set preparado:")
print(df_job_skill["job_id"].duplicated().sum())

print("Vagas com skills no Job Skill Set:")
print(df_job_skill["skills_job_skill_set"].fillna("").astype(str).str.strip().ne("").sum())

Colunas Job Skill Set preparado:
['job_id', 'category', 'job_title', 'title_normalized', 'job_description', 'skills_job_skill_set', 'skills_job_skill_set_lista']
Job IDs duplicados no Job Skill Set preparado:
0
Vagas com skills no Job Skill Set:
1167


## 3. Verificar possibilidade de composição por `job_id`

Antes do merge, verificamos quantos `job_id` existem nas duas bases.

In [8]:
relatorio_intersecao = verificar_intersecao_job_id(df_linkedin, df_job_skill)

for chave, valor in relatorio_intersecao.items():
    print(f"{chave}: {valor}")

qtd_vagas_linkedin: 123849
qtd_vagas_job_skill_set: 1167
qtd_job_ids_em_comum: 1167
percentual_linkedin_com_match: 0.94
percentual_job_skill_com_match: 100.0


## 4. Compor datasets

A composição usa o LinkedIn como base principal e adiciona informações do Job Skill Set quando o `job_id` aparece nas duas bases.

In [ ]:
df_vagas_enriquecidas, relatorio_composicao = compor_datasets_vagas(
    df_linkedin=df_linkedin,
    df_job_skill=df_job_skill
)

print("Formato da base final:", df_vagas_enriquecidas.shape)
display(df_vagas_enriquecidas.head())

In [ ]:
print("Relatório de composição")
print("=" * 70)

for chave, valor in relatorio_composicao.items():
    print(f"{chave}: {valor}")

## 5. Validar contrato da base final

A base final precisa atender ao contrato combinado com os outros desenvolvedores.

In [ ]:
print("Colunas obrigatórias esperadas:")
print(COLUNAS_BASE_FINAL)

print("Colunas da base final:")
print(df_vagas_enriquecidas.columns.tolist())

In [ ]:
validacao = validar_base_final(df_vagas_enriquecidas)

print("Validação da base final")
print("=" * 70)

for chave, valor in validacao.items():
    print(f"{chave}: {valor}")

In [ ]:
assert validacao["possui_colunas_obrigatorias"] == True
assert validacao["qtd_job_id_duplicados"] == 0
assert validacao["qtd_linhas"] > 0

print("Base final validada com sucesso.")

## 6. Análises rápidas da base final

Essas verificações ajudam a documentar a qualidade da composição.

In [ ]:
resumo = pd.DataFrame({
    "métrica": [
        "Quantidade de vagas",
        "Job IDs duplicados",
        "Vagas com texto completo",
        "Vagas com skills",
        "Vagas com empresa",
        "Vagas com salário mínimo",
        "Vagas com salário mediano",
        "Vagas com salário máximo",
    ],
    "valor": [
        len(df_vagas_enriquecidas),
        df_vagas_enriquecidas["job_id"].duplicated().sum(),
        df_vagas_enriquecidas["texto_vaga_completo"].fillna("").astype(str).str.strip().ne("").sum(),
        df_vagas_enriquecidas["skills"].fillna("").astype(str).str.strip().ne("").sum(),
        df_vagas_enriquecidas["company_name"].fillna("").astype(str).str.strip().ne("").sum(),
        df_vagas_enriquecidas["min_salary"].notna().sum(),
        df_vagas_enriquecidas["med_salary"].notna().sum(),
        df_vagas_enriquecidas["max_salary"].notna().sum(),
    ]
})

display(resumo)

In [ ]:
display(
    df_vagas_enriquecidas[[
        "job_id",
        "title",
        "company_name",
        "location",
        "skills",
        "texto_vaga_completo"
    ]].head(10)
)

## 7. Salvar base final local

A base completa será salva em `data/processed/vagas_enriquecidas.csv`.

Este arquivo **não deve ser versionado no GitHub**, pois fica dentro de `data/processed/`.

In [ ]:
caminho_base_final = salvar_base_final(
    df=df_vagas_enriquecidas,
    project_root=PROJECT_ROOT,
    nome_arquivo="vagas_enriquecidas.csv"
)

print("Base final salva em:")
print(caminho_base_final)

## 8. Criar amostra versionável

A amostra pequena será salva em `data/sample/vagas_exemplo.csv`.

Esse arquivo deve ser versionado no GitHub para que os outros desenvolvedores consigam testar recomendação, skills e interface sem depender da base completa.

In [ ]:
caminho_amostra = criar_amostra_versionavel(
    df=df_vagas_enriquecidas,
    project_root=PROJECT_ROOT,
    nome_arquivo="vagas_exemplo.csv",
    n=10
)

print("Amostra versionável salva em:")
print(caminho_amostra)

df_amostra = pd.read_csv(caminho_amostra)
display(df_amostra)

## 9. Conclusão da composição

A composição criou uma base final de vagas enriquecidas usando o LinkedIn Job Postings como catálogo principal e o Job Skill Set como fonte complementar de skills.

O merge foi feito por `job_id`, preservando as vagas do LinkedIn e adicionando skills externas quando disponíveis. A coluna `skills` consolida as habilidades vindas das bases disponíveis, enquanto `texto_vaga_completo` junta cargo, descrição e skills para ser usada pelos módulos de recomendação e similaridade textual.

A base completa foi salva localmente em `data/processed/vagas_enriquecidas.csv`, enquanto uma amostra pequena foi criada em `data/sample/vagas_exemplo.csv` para permitir que os outros desenvolvedores trabalhem em paralelo.

In [ ]:
df_tratado = df_vagas_enriquecidas.copy()

# Tratamento de textos básicos

df_tratado["company_name"] = (
    df_tratado["company_name"]
    .fillna("NOT_INFORMED")
    .astype(str)
)

df_tratado["description"] = (
    df_tratado["description"]
    .fillna("")
    .astype(str)
)

df_tratado["skills"] = (
    df_tratado["skills"]
    .fillna("")
    .astype(str)
)

df_tratado["formatted_experience_level"] = (
    df_tratado["formatted_experience_level"]
    .fillna("NOT_INFORMED")
    .astype(str)
)

df_tratado["pay_period"] = (
    df_tratado["pay_period"]
    .fillna("NOT_INFORMED")
    .astype(str)
)

# Tratamento de remoto

df_tratado["remote_status"] = np.where(
    df_tratado["remote_allowed"].fillna(0).astype(float) == 1,
    "REMOTE_ALLOWED",
    "NOT_INFORMED"
)

# Mantém a coluna original, mas também cria uma versão mais prática
df_tratado["remote_allowed_flag"] = np.where(
    df_tratado["remote_allowed"].fillna(0).astype(float) == 1,
    1,
    0
)

# Tratamento de salário

df_tratado["possui_salario"] = (
    df_tratado["min_salary"].notna()
    | df_tratado["med_salary"].notna()
    | df_tratado["max_salary"].notna()
)

# Mantém salários reais. Não preenche com média.
# Cria versões auxiliares para exibição.
df_tratado["salary_min_display"] = df_tratado["min_salary"]
df_tratado["salary_med_display"] = df_tratado["med_salary"]
df_tratado["salary_max_display"] = df_tratado["max_salary"]

# Quando só existe salário mediano, usa ele como referência visual
df_tratado["salary_min_display"] = df_tratado["salary_min_display"].fillna(df_tratado["med_salary"])
df_tratado["salary_max_display"] = df_tratado["salary_max_display"].fillna(df_tratado["med_salary"])

# Recriar texto final se necessário

df_tratado["texto_vaga_completo"] = (
    df_tratado["texto_vaga_completo"]
    .fillna("")
    .astype(str)
)

# Conferência final de nulos

print("Formato:", df_tratado.shape)

print("\nPercentual de nulos após tratamento:")
display((df_tratado.isna().mean() * 100).sort_values(ascending=False).round(2))

In [ ]:
def criar_faixa_salarial_texto(linha):
    if not linha["possui_salario"]:
        return "Não informado"

    min_salary = linha["salary_min_display"]
    med_salary = linha["salary_med_display"]
    max_salary = linha["salary_max_display"]
    pay_period = linha["pay_period"]

    if pd.notna(min_salary) and pd.notna(max_salary):
        return f"{min_salary:.2f} - {max_salary:.2f} ({pay_period})"

    if pd.notna(med_salary):
        return f"{med_salary:.2f} ({pay_period})"

    return "Não informado"


df_tratado["salary_range_text"] = df_tratado.apply(criar_faixa_salarial_texto, axis=1)

In [ ]:
try:
    df_tratado
except NameError:
    raise NameError("O DataFrame df_tratado não existe. Rode primeiro a célula de tratamento dos nulos.")

Path("data/processed").mkdir(parents=True, exist_ok=True)

# Cópia da base tratada
df_final = df_tratado.copy()

# Criar descrição curta para interface

df_final["description_preview"] = (
    df_final["description"]
    .fillna("")
    .astype(str)
    .str.slice(0, 800)
)

# Criar texto de faixa salarial para exibição

def criar_faixa_salarial_texto(linha):
    if "possui_salario" in linha.index and not linha["possui_salario"]:
        return "Não informado"

    min_salary = linha.get("salary_min_display", None)
    med_salary = linha.get("salary_med_display", None)
    max_salary = linha.get("salary_max_display", None)
    pay_period = linha.get("pay_period", "NOT_INFORMED")

    if pd.notna(min_salary) and pd.notna(max_salary):
        return f"{float(min_salary):.2f} - {float(max_salary):.2f} ({pay_period})"

    if pd.notna(med_salary):
        return f"{float(med_salary):.2f} ({pay_period})"

    return "Não informado"


df_final["salary_range_text"] = df_final.apply(criar_faixa_salarial_texto, axis=1)

# Salvar base completa tratada

df_final.to_csv(
    "data/processed/vagas_enriquecidas_tratada.csv",
    index=False
)

print("Base completa tratada salva em:")
print("data/processed/vagas_enriquecidas_tratada.csv")

# Criar versão final otimizada para uso no sistema

colunas_lite = [
    "job_id",
    "title",
    "company_name",
    "location",
    "skills",
    "texto_vaga_completo",
    "description_preview",
    "formatted_work_type",
    "formatted_experience_level",
    "remote_status",
    "remote_allowed_flag",
    "possui_salario",
    "salary_range_text",
    "pay_period",
    "job_posting_url"
]

# Mantém apenas colunas que existem no df_final
colunas_lite = [coluna for coluna in colunas_lite if coluna in df_final.columns]

df_lite = df_final[colunas_lite].copy()

# Salvar CSV otimizado
df_lite.to_csv(
    "data/processed/vagas_enriquecidas_lite.csv",
    index=False
)

# Salvar Parquet otimizado
df_lite.to_parquet(
    "data/processed/vagas_enriquecidas_lite.parquet",
    index=False
)

print("\nBase lite criada.")
print("Formato:", df_lite.shape)

print("\nTamanho CSV lite:")
print(round(os.path.getsize("data/processed/vagas_enriquecidas_lite.csv") / 1024**2, 2), "MB")

print("\nTamanho Parquet lite:")
print(round(os.path.getsize("data/processed/vagas_enriquecidas_lite.parquet") / 1024**2, 2), "MB")

print("\nUso aproximado em RAM:")
print(round(df_lite.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

print("\nColunas da base lite:")
print(df_lite.columns.tolist())

In [ ]:
Path("data/sample").mkdir(parents=True, exist_ok=True)

df_exemplo = df_lite.sample(
    n=50,
    random_state=42
).copy()

df_exemplo.to_csv(
    "data/sample/vagas_exemplo.csv",
    index=False
)

print("Novo vagas_exemplo.csv criado com as colunas atualizadas.")
print("Formato:", df_exemplo.shape)
print(df_exemplo.columns.tolist())

df_exemplo.head()